# 2270 Financial Modeling — Group Assignment
## Trend-filtered multi-asset ETF portfolio: backtest notebook

**Moodle group:** _[group name]_
**Members:** _[name 1, name 2, name 3, name 4, name 5]_

**Streamlit app:** https://financialmodelinggp-8mn85ltyh3prwzyudx7eqd.streamlit.app/
**GitHub repository:** https://github.com/cedrotwelve/FinancialModelingGP (app source code; `backtest_core.py` contains the same functions as this notebook)

---

### How to run
1. Open this notebook in a **fresh Google Colab session** (colab.research.google.com).
2. `Runtime → Run all`. No API key and no data files are needed: all data (ETF prices and the T-bill rate) come from the Yahoo Finance API through `yfinance`.
3. Every assumption is set **once**, in Section 1. This notebook produces every number, table and figure in the prospectus.
4. If Yahoo Finance rate-limits the download (an error in Section 2), wait a minute and run that cell again.

### The strategy in one paragraph
Four ETFs (SPY US equities, EFA developed ex-US equities, IEF 7–10y US Treasuries, GLD gold) start from 25% each. At every month-end, each asset whose price is below its 10-month simple moving average is moved to cash (3-month T-bills) until its price is back above the average (Faber, 2007). The signal observed at the end of month $t$ sets the weights for month $t+1$. Benchmarks: a monthly-rebalanced **60/40 SPY/IEF** portfolio for alpha, beta and the information ratio, and the **same 25% mix without the filter**, which shows what the rule itself adds.

### Structure
0. Setup · 1. Parameters · 2. Data · 3. Allocation rule · 4. Backtest · 5. Sanity checks · 6. Performance measures · 7. Figures · 8. Implementation: turnover, signals, costs · 9. Robustness: moving-average window · 10. Calculation assumptions · 11. AI disclosure

## 0. Setup

In [ ]:
# Install packages that are not pre-installed in Colab (yfinance usually is, quantstats is not)
import importlib.util, subprocess, sys
for pkg in ["yfinance", "quantstats", "statsmodels"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import yfinance as yf
import statsmodels.api as sm

try:
    import quantstats as qs                 # used only for the cross-check in Section 6
    HAS_QS = True
except Exception as e:
    HAS_QS = False
    print("quantstats not available, cross-check will be skipped:", e)

pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True

## 1. Parameters — single source of truth

The Streamlit app lets the user change `MA_WINDOW`, `TC_BPS` and `USE_FILTER`, and calls the same `run_all()` function as this notebook.

**Sample.** Returns run from January 2006 to December 2025 (240 months), a period that includes 2008, 2020 and 2022. Prices from 2005 are used only to compute the first moving averages: the first decision, at the end of December 2005, already has a complete 10-month window.

In [ ]:
# ---- Portfolio -----------------------------------------------------------------
STRATEGY_NAME  = "Trend-filtered 4-ETF"
STATIC_NAME    = "Static 25% mix (no filter)"
BENCHMARK_NAME = "60/40 SPY/IEF"

BASE_WEIGHTS  = {"SPY": 0.25, "EFA": 0.25, "IEF": 0.25, "GLD": 0.25}
BENCH_WEIGHTS = {"SPY": 0.60, "IEF": 0.40}
RF_TICKER     = "^IRX"            # 13-week US T-bill yield (% p.a.): return on cash and risk-free rate

# ---- Allocation rule -----------------------------------------------------------
MA_WINDOW  = 10                   # months in the simple moving average (Faber, 2007)
USE_FILTER = True                 # False -> static 25% mix
REBALANCE  = "M"                  # rebalance to target weights at every month-end

# ---- Sample (set explicitly) ---------------------------------------------------
DATA_START = "2005-01-01"         # warm-up: prices used only for the first moving averages
START      = "2006-01-01"         # first month of returns (first decision: end of Dec 2005)
END        = "2025-12-31"         # last month of returns (inclusive)
PERIODS_PER_YEAR = 12

# ---- Implementation ------------------------------------------------------------
TC_BPS       = 10                 # one-way cost, bp per unit of turnover (strategy and static mix)
BENCH_TC_BPS = 0                  # benchmark shown gross of costs (conservative for the strategy)
TC_GRID_BPS  = [0, 5, 10, 25, 50] # cost sensitivity (Section 8)
MA_GRID      = [6, 8, 10, 12]     # window sensitivity (Section 9); max 12 given the 2005 warm-up

# ---- Client and output ---------------------------------------------------------
CLIENT_PAIN_DD = 0.40             # drawdown at which the target client would sell in panic
ROLLING_WINDOW = 60               # months, rolling beta/alpha
SAVE_OUTPUTS   = True             # save figures and tables for the prospectus
OUTPUT_DIR     = "outputs"

## 2. Data

| | |
|---|---|
| **Source** | Yahoo Finance, through the `yfinance` API |
| **Assets** | SPY, EFA, IEF, GLD: adjusted close (`auto_adjust=True`), i.e. adjusted for splits and dividends, so returns are total returns |
| **Frequency** | daily closes reduced to the **last close of each month** (month-end prices) |
| **Risk-free / cash** | `^IRX`, 13-week T-bill yield in % p.a. The yield at month-end $t$ is converted to a monthly rate, $(1+y_t)^{1/12}-1$, and earned during month $t+1$ |
| **Sample** | prices Jan 2005 – Dec 2025; returns Jan 2006 – Dec 2025 |

**Back-adjusted prices and look-ahead.** Yahoo adjusts *past* prices for dividends paid later. This rescales all prices before a dividend by the same factor, so ratios of past prices (returns, price vs moving average) are unaffected. The signal therefore uses no future information.

In [ ]:
def download_monthly_prices(tickers, start, end, retries=3, pause=5):
    """Daily adjusted closes from Yahoo Finance (auto_adjust=True: split- and dividend-adjusted)
    -> last available close of each calendar month. `end` is INCLUSIVE here
    (yfinance's own `end` argument is exclusive)."""
    tickers = [tickers] if isinstance(tickers, str) else list(tickers)
    end_excl = (pd.Timestamp(end) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    missing = tickers
    for attempt in range(retries):
        try:
            raw = yf.download(tickers, start=start, end=end_excl, interval="1d",
                              auto_adjust=True, progress=False)
            px = raw["Close"]
        except Exception as e:                       # network error / rate limit
            print(f"Download attempt {attempt + 1} failed: {e}")
            px = pd.DataFrame(columns=tickers)
        if isinstance(px, pd.Series):
            px = px.to_frame(name=tickers[0])
        missing = [t for t in tickers if t not in px.columns or px[t].isna().all()]
        if not missing:
            break
        if attempt + 1 < retries:
            print(f"No data for {missing}, retrying in {pause}s ...")
            time.sleep(pause)
    if missing:
        raise RuntimeError(f"Yahoo Finance returned no data for {missing}. "
                           "Re-run the cell (Yahoo rate limits are common).")
    px = px[tickers]
    if px.index.tz is not None:
        px = px.tz_localize(None)
    monthly = px.groupby(px.index.to_period("M")).last()          # last close of each month
    monthly.index = monthly.index.to_timestamp(how="end").normalize()
    monthly.index.name = "date"
    return monthly


def rf_from_irx(irx_pct, periods_per_year=12):
    """^IRX is the 13-week T-bill yield in % p.a. The yield observed at month-end t is the
    (known) return on cash during month t+1, converted to a monthly rate."""
    y = irx_pct.ffill() / 100
    return ((1 + y) ** (1 / periods_per_year) - 1).shift(1).rename("rf")

In [ ]:
raw_px = download_monthly_prices(list(BASE_WEIGHTS) + [RF_TICKER], DATA_START, END)
prices = raw_px[list(BASE_WEIGHTS)]          # asset prices
irx    = raw_px[RF_TICKER]                   # T-bill yield, % p.a.
rf_ret = rf_from_irx(irx, PERIODS_PER_YEAR)  # monthly return on cash, for month t+1

### Data checks (shape, date range, missing values, plots)

In [ ]:
print("1. Shape (months x assets):", prices.shape)
print(f"2. Date range: {prices.index[0]:%Y-%m-%d} -> {prices.index[-1]:%Y-%m-%d}")
print("3. Missing values per series:")
display(raw_px.isna().sum().to_frame("missing"))
print(f"   ^IRX range: {irx.min():.2f}% to {irx.max():.2f}% (median {irx.median():.2f}%)")
if irx.max() > 25 or irx.min() < -1:
    print("   WARNING: ^IRX outside a plausible range for a % yield, check the units!")

In [ ]:
# 4. Plots: rebased prices (log scale) and the T-bill rate
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
(prices / prices.iloc[0]).plot(ax=a1, logy=True, title="Month-end prices, rebased to 1 (log scale)")
irx.plot(ax=a2, color="grey", title="^IRX: 3-month T-bill yield (% p.a.)")
plt.tight_layout(); plt.show()

## 3. Allocation rule

The rule is a function that receives **only prices up to and including the decision date** and returns target weights. Weights that do not sum to 1 leave the remainder in cash, which earns the T-bill rate.

$$w_{i,t} = 0.25 \times \mathbb{1}\left\{P_{i,t} > \tfrac{1}{10}\textstyle\sum_{k=0}^{9} P_{i,t-k}\right\}, \qquad w_{cash,t} = 1 - \textstyle\sum_i w_{i,t}$$

The 60/40 benchmark uses the same engine with fixed weights, rebalanced monthly.

In [ ]:
def make_fixed_rule(weights):
    """Constant target weights (used for the 60/40 benchmark)."""
    w = pd.Series(weights, dtype=float)
    def rule(prices_hist):
        return w.copy()
    return rule


def make_trend_rule(base_weights, window=10, use_filter=True):
    """Faber (2007)-style trend filter. At each month-end, an asset keeps its base weight if its
    price is above its `window`-month simple moving average; otherwise its weight goes to cash.
    `prices_hist` contains prices up to and including the decision date only."""
    base = pd.Series(base_weights, dtype=float)
    def rule(prices_hist):
        if not use_filter:
            return base.copy()
        p = prices_hist[base.index].iloc[-window:]
        full = (len(p) == window) & p.notna().all()     # require a complete window
        above = p.iloc[-1] > p.mean()                    # month-end price vs moving average
        return base.where(above & full, 0.0)
    return rule

In [ ]:
strategy_rule = make_trend_rule(BASE_WEIGHTS, MA_WINDOW, USE_FILTER)
print("Target weights at the first decision date:")
print(strategy_rule(prices.loc[:pd.Timestamp(START) - pd.Timedelta(days=1)]))

## 4. Backtest engine

- **Timing (no look-ahead).** Weights decided with data up to month-end $t$ earn the return of month $t+1$. This is the `.shift(1)` convention of Lecture 4.
- **Returns.** Simple returns, so the portfolio return is exactly $R_{p,t+1}=\sum_i w_{i,t}R_{i,t+1}+w_{cash,t}R^f_{t+1}$. With log returns this sum would only be an approximation.
- **Rebalancing and drift.** At every month-end the portfolio returns to its target weights. Within the month, weights drift with prices.
- **Turnover and costs.** Turnover is $\sum_i |w^{target}_i - w^{drifted}_i|$ over the risky assets. The cost is turnover $\times$ `TC_BPS` and reduces the capital invested in the next month: $1+R^{net}=(1+R^{gross})(1-\text{cost})$. The initial purchase counts as turnover. Cash is assumed to be held at no trading cost.

In [ ]:
def rebalance_flags(dates, rule):
    """True on the dates where the portfolio is rebalanced (always on the first date)."""
    flags = np.zeros(len(dates), dtype=bool)
    if rule != "BH":
        freq = {"M": "M", "Q": "Q", "A": "Y"}[rule]
        per = np.asarray(dates.to_period(freq))
        flags[:-1] = per[:-1] != per[1:]               # last observation of each period
    flags[0] = True                                    # initial investment
    return pd.Series(flags, index=dates)


def run_backtest(prices, weight_fn, rebalance="M", tc_bps=0.0, rf=None, start=None):
    """Event-driven monthly backtest.
    prices : month-end prices (may include a warm-up period before `start`)
    start  : first month whose return is recorded. The first decision is taken at the last
             month-end before `start`, using data up to that date only.
    rf     : per-period return earned by the cash weight, indexed by return date
    Returns (res, weights): res has gross/net returns, turnover and cost per return date;
    weights are the asset weights HELD during each return period (decided one period earlier)."""
    rets  = prices.pct_change(fill_method=None)
    dates = prices.index
    i0 = 0
    if start is not None:
        i0 = int(dates.searchsorted(pd.Timestamp(start))) - 1
        if i0 < 0:
            raise ValueError("No month-end before `start`: cannot take the first decision.")
    trade_dates = dates[i0:]
    flags = rebalance_flags(trade_dates, rebalance)
    tc    = tc_bps / 10_000
    w     = pd.Series(0.0, index=prices.columns)       # start in cash
    rows, held, warn = [], [], []

    for k in range(len(trade_dates) - 1):
        t, t_next = trade_dates[k], trade_dates[k + 1]
        turnover = 0.0
        if flags.iloc[k]:
            target = weight_fn(prices.loc[:t]).reindex(prices.columns).fillna(0.0)
            turnover = float((target - w).abs().sum())
            w = target

        r = rets.loc[t_next]
        missing = (w != 0) & r.isna()
        if missing.any():                              # held asset without a return: flag it
            warn.append((t_next.strftime("%Y-%m"), list(r.index[missing])))
        r = r.fillna(0.0)

        rf_t = 0.0 if rf is None else rf.get(t_next, np.nan)
        if pd.isna(rf_t):
            warn.append((t_next.strftime("%Y-%m"), "risk-free missing"))
            rf_t = 0.0
        cash_w = 1.0 - float(w.sum())
        gross  = float((w * r).sum() + cash_w * rf_t)
        cost   = turnover * tc                          # paid at the trade, at month-end t
        net    = (1 + gross) * (1 - cost) - 1

        rows.append((t_next, gross, net, turnover, cost))
        held.append(w.rename(t_next))
        w = w * (1 + r) / (1 + gross)                  # weights drift until the next rebalance

    if warn:
        print(f"WARNING: {len(warn)} data issues (returns set to 0):", warn[:5])
    res = pd.DataFrame(rows, columns=["date", "gross", "net", "turnover", "cost"]).set_index("date")
    return res, pd.DataFrame(held)


def run_all(prices, rf, base_weights, bench_weights, ma_window=10, use_filter=True,
            tc_bps=10.0, bench_tc_bps=0.0, rebalance="M", start=None):
    """Runs the strategy, the unfiltered static mix and the benchmark on the same sample.
    This is the single entry point used by both the notebook and the Streamlit app."""
    if use_filter and start is not None:
        n_hist = int(prices.index.searchsorted(pd.Timestamp(start)))   # month-ends up to the first decision
        if n_hist < ma_window:
            raise ValueError(f"A {ma_window}-month moving average needs {ma_window} month-ends before {start}, "
                             f"but the data have {n_hist}. Start the download earlier or shorten the window.")
    strat, weights = run_backtest(prices, make_trend_rule(base_weights, ma_window, use_filter),
                                  rebalance, tc_bps, rf, start)
    static, _ = run_backtest(prices, make_trend_rule(base_weights, ma_window, False),
                             rebalance, tc_bps, rf, start)
    bench, _ = run_backtest(prices, make_fixed_rule(bench_weights),
                            rebalance, bench_tc_bps, rf, start)
    return {"strategy": strat, "weights": weights, "static": static,
            "benchmark": bench, "rf": rf.reindex(strat.index)}

In [ ]:
out = run_all(prices, rf_ret, BASE_WEIGHTS, BENCH_WEIGHTS, MA_WINDOW, USE_FILTER,
              TC_BPS, BENCH_TC_BPS, REBALANCE, START)
res, weights = out["strategy"], out["weights"]
strat, strat_gross = res["net"], res["gross"]
static, bench, rf  = out["static"]["net"], out["benchmark"]["net"], out["rf"]

assert not pd.concat([strat, static, bench, rf], axis=1).isna().any().any(), "missing values in the sample"
print(f"Return sample: {strat.index[0]:%b %Y} -> {strat.index[-1]:%b %Y} ({len(strat)} months)")
res.head()

## 5. Sanity checks

**(a) Look-ahead test.** All prices *after* a cut-off date are scrambled, and the backtest is run again. If the rule uses only past data, every weight decided up to the cut-off must be unchanged. To show that the test works, it is also run on a deliberately wrong rule that peeks at next month's prices; the test must flag that rule.

**(b) Cross-check against the vectorized formula.** With monthly rebalancing, the engine must match the one-line `shift(1)` formula of Lecture 4: signal at $t$, weights shifted by one month, multiplied by the returns of $t+1$.

In [ ]:
def lookahead_test(rule, prices_in, rebalance, start, n_trials=5):
    """Scramble prices after a cut-off and check that weights decided up to the cut-off are unchanged.
    The global `prices` is swapped too, so a rule that reads it instead of its argument is caught."""
    idx = prices_in.index
    first = idx.searchsorted(pd.Timestamp(start)) - 1
    flags = np.zeros(len(idx), dtype=bool)
    flags[first:] = rebalance_flags(idx[first:], rebalance).values
    cut = idx[flags & (idx <= idx[(first + len(idx)) // 2])][-1]       # a decision date mid-sample
    held_next = idx[idx.get_loc(cut) + 1]                                # weights held in t+1 decided at t
    _, w_orig = run_backtest(prices_in, rule, rebalance, 0.0, start=start)
    keep = w_orig.index <= held_next
    saved = globals().get("prices")
    try:
        for seed in range(n_trials):
            rng = np.random.default_rng(seed)
            scrambled = prices_in.copy()
            after = idx > cut
            scrambled.loc[after] = scrambled.loc[after] * rng.uniform(0.5, 1.5, size=scrambled.loc[after].shape)
            globals()["prices"] = scrambled
            _, w_scr = run_backtest(scrambled, rule, rebalance, 0.0, start=start)
            if not np.allclose(w_orig.loc[keep].values, w_scr.loc[keep].values):
                return False, cut
    finally:
        globals()["prices"] = saved
    return True, cut


ok, cut = lookahead_test(strategy_rule, prices, REBALANCE, START)
print(f"(a) Look-ahead test, strategy rule (cut-off {cut:%Y-%m}):",
      "PASSED" if ok else "FAILED: the rule uses future data!")

def _cheating_rule(prices_hist):
    """WRONG ON PURPOSE: holds only the assets that will rise next month."""
    i = prices.index.get_loc(prices_hist.index[-1])
    base = pd.Series(BASE_WEIGHTS)
    if i + 1 >= len(prices):
        return base
    next_ret = prices[base.index].iloc[i + 1] / prices[base.index].iloc[i] - 1
    return base.where(next_ret > 0, 0.0)

ok_cheat, _ = lookahead_test(_cheating_rule, prices, REBALANCE, START)
print("    Same test on a deliberately cheating rule:",
      "caught (FAILED), as it should be" if not ok_cheat else "NOT caught: the test is not working")

In [ ]:
# (b) vectorized cross-check (valid with monthly rebalancing)
if REBALANCE == "M":
    base = pd.Series(BASE_WEIGHTS)
    rets = prices.pct_change(fill_method=None)[base.index]
    if USE_FILTER:
        signal = (prices[base.index] > prices[base.index].rolling(MA_WINDOW).mean()).astype(float)
    else:
        signal = pd.DataFrame(1.0, index=prices.index, columns=base.index)
    w_vec = (signal * base).shift(1)                                      # decided at t, held in t+1
    r_vec = (w_vec * rets).sum(axis=1) + (1 - w_vec.sum(axis=1)) * rf_ret
    diff = (r_vec.reindex(strat.index) - strat_gross).abs().max()
    print(f"(b) Max abs. difference, engine vs vectorized formula: {diff:.2e}",
          "-> OK" if diff < 1e-10 else "-> CHECK")
else:
    print("(b) Skipped: the vectorized formula assumes monthly rebalancing.")

## 6. Performance measures

All measures use monthly simple returns, Jan 2006 – Dec 2025. *Excess* means in excess of the T-bill rate. Alpha, beta and the information ratio are measured against the **60/40 SPY/IEF** benchmark.

| Measure | Definition | Annualization |
|---|---|---|
| Avg. return | arithmetic mean $\bar R$ | $\times 12$ |
| CAGR | $\left(\prod (1+R_t)\right)^{12/T}-1$ | already annual |
| Volatility | $\sigma(R)$ | $\times\sqrt{12}$ |
| Sharpe ratio | $\overline{R-R^f}\,/\,\sigma(R-R^f)$ | $\times\sqrt{12}$ |
| Max drawdown | $\min_t (W_t / \max_{s\le t} W_s - 1)$ | — |
| Worst 12-month return | lowest compounded return over any 12 consecutive months | — |
| Beta | OLS slope in $R^{ex}_t=\alpha+\beta R^{b,ex}_t+\varepsilon_t$ | **not annualized** |
| Alpha | OLS intercept (t-stat: OLS standard errors) | $\times 12$ |
| Information ratio (regression) | $\alpha / \sigma(\varepsilon)$ | $\times\sqrt{12}$ |
| Tracking error | $\sigma(R-R^b)$ | $\times\sqrt{12}$ |
| Information ratio (TE) | $\overline{R-R^b}\,/\,\sigma(R-R^b)$ | $\times\sqrt{12}$ |

Means scale with time, volatilities with the square root of time. Standard deviations use $T-1$ degrees of freedom.

In [ ]:
def perf_stats(r, rf, bench=None, ppy=12):
    """Performance measures of a periodic simple-return series (definitions in the notebook)."""
    ex = r - rf
    wealth = (1 + r).cumprod()
    out = {
        "Avg. return (ann.)":     r.mean() * ppy,
        "CAGR":                   wealth.iloc[-1] ** (ppy / len(r)) - 1,
        "Volatility (ann.)":      r.std() * np.sqrt(ppy),
        "Sharpe ratio (ann.)":    ex.mean() * ppy / (ex.std() * np.sqrt(ppy)),
        "Max drawdown":           (wealth / wealth.cummax() - 1).min(),
        "Worst 12-month return":  (1 + r).rolling(ppy).apply(np.prod, raw=True).min() - 1,
    }
    if bench is not None:
        bex = bench - rf
        fit = sm.OLS(ex.values, sm.add_constant(bex.values)).fit()
        alpha, beta = fit.params
        sigma_eps = np.sqrt(fit.mse_resid)
        active = r - bench
        out.update({
            "Beta":                     beta,
            "Alpha (ann.)":             alpha * ppy,
            "Alpha t-stat":             fit.tvalues[0],
            "R-squared":                fit.rsquared,
            "Info. ratio (regression)": alpha / sigma_eps * np.sqrt(ppy),
            "Tracking error (ann.)":    active.std() * np.sqrt(ppy),
            "Info. ratio (TE)":         active.mean() * ppy / (active.std() * np.sqrt(ppy)),
        })
    return pd.Series(out)


PCT_ROWS = ["Avg. return (ann.)", "CAGR", "Volatility (ann.)", "Max drawdown",
            "Worst 12-month return", "Alpha (ann.)", "Tracking error (ann.)",
            "Annual turnover", "Avg. cash weight", "Cost drag (ann.)"]

def format_table(t, pct_rows=PCT_ROWS):
    """Percentages for return-type rows, 2 decimals for ratios."""
    f = t.copy().astype(object)
    for row in t.index:
        f.loc[row] = [("" if pd.isna(v) else (f"{v:.2%}" if row in pct_rows else f"{v:.2f}"))
                      for v in t.loc[row]]
    return f


def cum_index(r, start_date):
    """Cumulative value of 1 invested at start_date (the series starts at exactly 1)."""
    return pd.concat([pd.Series([1.0], index=[start_date]), (1 + r).cumprod()])


def calendar_returns(r):
    """Compounded return per calendar year."""
    return (1 + r).groupby(r.index.year).prod() - 1

In [ ]:
stats = pd.DataFrame({
    STRATEGY_NAME:  perf_stats(strat,  rf, bench, PERIODS_PER_YEAR),
    STATIC_NAME:    perf_stats(static, rf, bench, PERIODS_PER_YEAR),
    BENCHMARK_NAME: perf_stats(bench,  rf, None,  PERIODS_PER_YEAR),
})
stats = stats.reindex(perf_stats(strat, rf, bench, PERIODS_PER_YEAR).index)
stats_fmt = format_table(stats)
print(f"Net of {TC_BPS} bp costs (strategy and static mix); benchmark gross. Jan 2006 - Dec 2025.")
display(stats_fmt)
print(f"Blank cells for {BENCHMARK_NAME}: it is the reference portfolio, so its beta is 1 and its alpha 0 "
      "by definition, and the information ratio is not defined.")

**Cross-check with `quantstats`.** The hand-computed values should match the library, as in Lecture 3. `quantstats`' information ratio is *not* annualized, so it is compared with our monthly value.

In [ ]:
if HAS_QS:
    ex, bex, act = strat - rf, bench - rf, strat - bench
    rows = {}
    try:
        rows["Sharpe (ann.)"] = (stats.loc["Sharpe ratio (ann.)", STRATEGY_NAME],
                                 qs.stats.sharpe(ex, rf=0.0, periods=PERIODS_PER_YEAR, annualize=True))
    except Exception as e: print("sharpe:", e)
    try:
        g = qs.stats.greeks(ex, bex, periods=PERIODS_PER_YEAR)
        rows["Beta"] = (stats.loc["Beta", STRATEGY_NAME], g["beta"])
        rows["Alpha (ann.)"] = (stats.loc["Alpha (ann.)", STRATEGY_NAME], g["alpha"])
    except Exception as e: print("greeks:", e)
    try:
        rows["Info. ratio TE (monthly)"] = (act.mean() / act.std(), qs.stats.information_ratio(strat, bench))
    except Exception as e: print("information_ratio:", e)
    display(pd.DataFrame(rows, index=["manual", "quantstats"]).T)
else:
    print("quantstats not installed: cross-check skipped.")

### Calendar-year returns (the years that matter for this client: 2008, 2020, 2022)

In [ ]:
cal = pd.DataFrame({STRATEGY_NAME: calendar_returns(strat),
                    STATIC_NAME: calendar_returns(static),
                    BENCHMARK_NAME: calendar_returns(bench)})
cal["Strategy - static"] = cal[STRATEGY_NAME] - cal[STATIC_NAME]
display(cal.apply(lambda col: col.map(lambda v: f"{v:.1%}")))

## 7. Figures

In [ ]:
if SAVE_OUTPUTS:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

def save(fig, name):
    if SAVE_OUTPUTS:
        fig.savefig(os.path.join(OUTPUT_DIR, name), dpi=200, bbox_inches="tight")

COLORS = {STRATEGY_NAME: "C0", STATIC_NAME: "C1", BENCHMARK_NAME: "k"}
start_date = prices.index[prices.index.get_loc(strat.index[0]) - 1]     # Dec 2005 = 1
cum = pd.DataFrame({STRATEGY_NAME: cum_index(strat, start_date),
                    STATIC_NAME: cum_index(static, start_date),
                    BENCHMARK_NAME: cum_index(bench, start_date)})

fig, ax = plt.subplots()
for c in cum:
    ax.plot(cum.index, cum[c], color=COLORS[c], lw=2.2 if c == STRATEGY_NAME else 1.6, label=c)
ax.axhline(1, color="grey", lw=0.8)
ax.set_title(f"Cumulative performance, {start_date:%b %Y} = 1 (strategy and static mix net of {TC_BPS} bp costs)")
ax.set_ylabel("Value of 1 invested"); ax.legend(loc="upper left")
plt.tight_layout(); save(fig, "fig1_cumulative.png"); plt.show()

In [ ]:
dd = cum / cum.cummax() - 1
fig, ax = plt.subplots(figsize=(11, 4))
for c in dd:
    ax.plot(dd.index, dd[c], color=COLORS[c], lw=1.8 if c == STRATEGY_NAME else 1.3, label=c)
ax.axhline(-CLIENT_PAIN_DD, color="red", ls="--", lw=1, label=f"client's pain threshold (-{CLIENT_PAIN_DD:.0%})")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylim(min(dd.min().min(), -CLIENT_PAIN_DD) - 0.05, 0.02)
ax.set_title("Drawdowns from the previous peak")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=4, fontsize=9, frameon=False)
plt.tight_layout(); save(fig, "fig2_drawdowns.png"); plt.show()

In [ ]:
# what the portfolio actually held, month by month
# (weights are indexed by the month-end at which the holding month ENDS)
cols   = list(weights.columns)
left   = weights.index - pd.offsets.MonthEnd(1)                 # start of each holding month
width  = np.asarray((weights.index - left).days, dtype=float)
palette = ["C0", "C2", "C4", "goldenrod", "C1", "C3", "C5", "C6"]
fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 4.8), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
for i, a in enumerate(cols[::-1]):                               # first asset on top
    held_m = np.asarray(weights[a] > 0)
    a1.bar(left[held_m], 0.76, width=width[held_m], bottom=i - 0.38, align="edge",
           color=palette[cols.index(a) % len(palette)], linewidth=0)
a1.set_yticks(range(len(cols))); a1.set_yticklabels(cols[::-1]); a1.set_ylim(-0.6, len(cols) - 0.4)
a1.grid(False, axis="y")
a1.set_title(f"Months in which each asset is held (coloured) or moved to cash (blank), {MA_WINDOW}-month filter")
cash_w = 1 - weights.sum(axis=1)
a2.bar(left, cash_w.values, width=width, align="edge", color="grey", alpha=0.7, linewidth=0)
a2.set_ylim(0, 1); a2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
a2.set_title("Weight in cash (T-bills)")
plt.tight_layout(); save(fig, "fig3_allocation.png"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
cal[[STRATEGY_NAME, STATIC_NAME, BENCHMARK_NAME]].plot(kind="bar", ax=ax, width=0.8,
        color=[COLORS[STRATEGY_NAME], COLORS[STATIC_NAME], "grey"])
ax.axhline(0, color="k", lw=0.8); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title("Calendar-year returns"); ax.set_xlabel(""); ax.legend(fontsize=9)
plt.tight_layout(); save(fig, "fig4_calendar_returns.png"); plt.show()

In [ ]:
ex, bex = strat - rf, bench - rf
roll_beta  = ex.rolling(ROLLING_WINDOW).cov(bex) / bex.rolling(ROLLING_WINDOW).var()
roll_alpha = (ex.rolling(ROLLING_WINDOW).mean() - roll_beta * bex.rolling(ROLLING_WINDOW).mean()) * PERIODS_PER_YEAR
fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
a1.plot(roll_beta, color="C0"); a1.axhline(1, color="grey", lw=0.8)
a1.set_title(f"Rolling {ROLLING_WINDOW}-month beta vs {BENCHMARK_NAME}")
a2.plot(roll_alpha, color="C2"); a2.axhline(0, color="grey", lw=0.8)
a2.set_title(f"Rolling {ROLLING_WINDOW}-month alpha vs {BENCHMARK_NAME} (annualized)")
a2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.tight_layout(); save(fig, "fig5_rolling_beta_alpha.png"); plt.show()

## 8. Implementation: turnover, signals and transaction costs

Turnover excludes the initial purchase. A **switch** is a month in which an asset moves into cash or back into the market. Frequent switches with little benefit are the *false signals* (whipsaws) described in the risk section.

In [ ]:
def implementation_stats(res_df, w_df=None, ppy=PERIODS_PER_YEAR):
    s = {"Annual turnover": res_df["turnover"].iloc[1:].mean() * ppy,
         "Cost drag (ann.)": res_df["cost"].iloc[1:].mean() * ppy}
    if w_df is not None:
        s["Avg. cash weight"] = (1 - w_df.sum(axis=1)).mean()
    return pd.Series(s)

impl = pd.DataFrame({STRATEGY_NAME: implementation_stats(res, weights),
                     STATIC_NAME: implementation_stats(out["static"])})
display(format_table(impl))

invested = weights > 0
signals = pd.DataFrame({
    "% of months invested": invested.mean(),
    "switches (in or out)": invested.astype(int).diff().abs().iloc[1:].sum(),
})
signals["switches per year"] = signals["switches (in or out)"] / (len(invested) / PERIODS_PER_YEAR)
display(pd.DataFrame({"% of months invested": signals["% of months invested"].map(lambda v: f"{v:.0%}"),
                      "switches (in or out)": signals["switches (in or out)"].map(lambda v: f"{v:.0f}"),
                      "switches per year":    signals["switches per year"].map(lambda v: f"{v:.1f}")}))

In [ ]:
# cost sensitivity: how much of the result survives higher trading costs?
sens = {}
for c in TC_GRID_BPS:
    o = run_all(prices, rf_ret, BASE_WEIGHTS, BENCH_WEIGHTS, MA_WINDOW, USE_FILTER, c, BENCH_TC_BPS, REBALANCE, START)
    s_t = perf_stats(o["strategy"]["net"], o["rf"], o["benchmark"]["net"], PERIODS_PER_YEAR)
    s_s = perf_stats(o["static"]["net"],   o["rf"], o["benchmark"]["net"], PERIODS_PER_YEAR)
    sens[f"{c} bp"] = {"Strategy: avg. return (ann.)": s_t["Avg. return (ann.)"],
                       "Strategy: Sharpe ratio":       s_t["Sharpe ratio (ann.)"],
                       "Strategy: alpha (ann.)":       s_t["Alpha (ann.)"],
                       "Static mix: Sharpe ratio":     s_s["Sharpe ratio (ann.)"]}
sens = pd.DataFrame(sens)
display(format_table(sens, pct_rows=["Strategy: avg. return (ann.)", "Strategy: alpha (ann.)"]))

## 9. Robustness: the moving-average window

The 10-month window comes from the literature, which was itself written after studying past data. If the results depended heavily on exactly 10 months, the rule would be fragile, and its past performance would partly reflect a choice made with hindsight. The table repeats the backtest with other windows, all net of `TC_BPS`.

In [ ]:
rob = {}
for m in MA_GRID:
    o = run_all(prices, rf_ret, BASE_WEIGHTS, BENCH_WEIGHTS, m, True, TC_BPS, BENCH_TC_BPS, REBALANCE, START)
    rob[f"{m}-month MA"] = pd.concat([perf_stats(o["strategy"]["net"], o["rf"], o["benchmark"]["net"], PERIODS_PER_YEAR),
                                      implementation_stats(o["strategy"], o["weights"])])
rob["No filter"] = pd.concat([perf_stats(static, rf, bench, PERIODS_PER_YEAR), implementation_stats(out["static"])])
rob = pd.DataFrame(rob).loc[["Avg. return (ann.)", "Volatility (ann.)", "Sharpe ratio (ann.)", "Max drawdown",
                             "Worst 12-month return", "Alpha (ann.)", "Info. ratio (regression)", "Annual turnover"]]
display(format_table(rob))

In [ ]:
if SAVE_OUTPUTS:
    for name, t in {"table_performance": stats_fmt, "table_performance_raw": stats, "table_calendar": cal,
                    "table_implementation": impl, "table_signals": signals,
                    "table_cost_sensitivity": sens, "table_ma_robustness": rob}.items():
        t.to_csv(os.path.join(OUTPUT_DIR, f"{name}.csv"))
    print("Saved to", os.path.abspath(OUTPUT_DIR), ":", sorted(os.listdir(OUTPUT_DIR)))

## 10. Calculation assumptions (for the prospectus)

This cell writes out the assumptions that the prospectus must state. The text is generated from the parameters, so it always matches the numbers above.

In [ ]:
print(f"""Data: Yahoo Finance via yfinance; daily adjusted closes (splits and dividends, auto_adjust=True)
  reduced to month-end prices. Cash and risk-free: {RF_TICKER} (13-week T-bill yield), month-end value converted to
  a monthly rate (1+y)^(1/12)-1 and earned in the following month.
Sample: monthly returns {strat.index[0]:%B %Y} - {strat.index[-1]:%B %Y} ({len(strat)} months); prices from {DATA_START[:4]} used
  only to compute the first moving averages.
Strategy: {', '.join(f'{k} {v:.0%}' for k, v in BASE_WEIGHTS.items())}; an asset below its {MA_WINDOW}-month simple moving
  average at month-end t is held in cash during month t+1. Rebalanced to target weights every month-end.
Benchmarks: {BENCHMARK_NAME} ({', '.join(f'{k} {v:.0%}' for k, v in BENCH_WEIGHTS.items())}), rebalanced monthly, gross of costs;
  {STATIC_NAME}, net of the same costs as the strategy.
Returns: simple returns; weights drift within the month.
Transaction costs: {TC_BPS} bp per unit of turnover (initial purchase included); cash held at no cost.
Annualization: means x{PERIODS_PER_YEAR}, volatilities x sqrt({PERIODS_PER_YEAR}). Sharpe ratio, alpha and information ratio
  use excess returns over the T-bill rate. Beta and alpha come from an OLS regression of the strategy's excess returns
  on the benchmark's excess returns; beta is not annualized, alpha is x{PERIODS_PER_YEAR}.""")

## 11. Generative-AI disclosure

_[To be completed by the group: tools used, purpose (e.g. code structure, debugging, wording), and how outputs were checked: data checks, look-ahead test with a deliberately cheating rule, cross-check against the vectorized `shift(1)` formula, cross-check against `quantstats`, plausibility of the results.]_